# IBCS scaling rules - YOLO starter for your GitHub repo

Dit notebook is aangepast op jouw huidige GitHub-structuur:

```text
IBCS-scaling-rules-check-YOLO/
├── Dataset/
│   ├── Compliant/
│   └── Not Compliant/
└── yolo.ipynb
```

## Belangrijk
Met deze dataset kun je **nu al** beginnen met een **classification model**:

- Compliant
- Not Compliant

Dat is iets anders dan echte **object detection**.

### Waarom?
Voor object detection heb je per afbeelding ook **bounding boxes / annotaties** nodig.
Die zie ik in deze repo nu nog niet terug.

Dus op dit moment is de logischste start:
1. eerst een classification model trainen
2. daarna pas uitbreiden naar detection als jullie ook dashboard-onderdelen gaan annoteren

## Wat dit notebook doet
- dataset tellen
- train/val/test splits maken
- YOLO classification trainen
- valideren
- voorspellingen maken
- confusion matrix maken
- simpele uitleg printen

In [ ]:
# Alleen nodig als je deze libraries nog niet hebt
# !pip install -U ultralytics scikit-learn matplotlib pandas pillow

In [ ]:
from pathlib import Path
import random
import shutil
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from ultralytics import YOLO

## 1. Paden instellen

In [ ]:
# PAS DIT AAN ALS JE PROJECT ELDERS STAAT
PROJECT_ROOT = Path.cwd()

DATASET_DIR = PROJECT_ROOT / "Dataset"
COMPLIANT_DIR = DATASET_DIR / "Compliant"
NON_COMPLIANT_DIR = DATASET_DIR / "Not Compliant"

SPLIT_DIR = PROJECT_ROOT / "dataset_split"

MODEL_NAME = "yolov8n-cls.pt"
IMG_SIZE = 640
EPOCHS = 20
BATCH = 16
DEVICE = 0   # gebruik "cpu" als je geen GPU hebt
WORKERS = 2

## 2. Dataset check

In [ ]:
def get_images(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return [p for p in Path(folder).glob("*") if p.suffix.lower() in exts]

compliant_images = get_images(COMPLIANT_DIR)
non_compliant_images = get_images(NON_COMPLIANT_DIR)

print("Compliant:", len(compliant_images))
print("Not Compliant:", len(non_compliant_images))
print("Totaal:", len(compliant_images) + len(non_compliant_images))

In [ ]:
def preview_images(image_list, title, n=6):
    if not image_list:
        print(f"Geen afbeeldingen gevonden voor {title}")
        return

    sample = random.sample(image_list, min(n, len(image_list)))

    plt.figure(figsize=(14, 8))
    for i, img_path in enumerate(sample, start=1):
        img = Image.open(img_path)
        plt.subplot(2, 3, i)
        plt.imshow(img)
        plt.title(img_path.name, fontsize=9)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

preview_images(compliant_images, "Compliant voorbeelden")
preview_images(non_compliant_images, "Not Compliant voorbeelden")

## 3. Train / val / test split maken

YOLO classification verwacht deze structuur:

```text
dataset_split/
├── train/
│   ├── compliant/
│   └── not_compliant/
├── val/
│   ├── compliant/
│   └── not_compliant/
└── test/
    ├── compliant/
    └── not_compliant/
```

In [ ]:
def prepare_split_dirs(base_dir):
    for split in ["train", "val", "test"]:
        for cls in ["compliant", "not_compliant"]:
            (base_dir / split / cls).mkdir(parents=True, exist_ok=True)

def clear_split_dir(base_dir):
    if base_dir.exists():
        shutil.rmtree(base_dir)
    base_dir.mkdir(parents=True, exist_ok=True)

def split_and_copy(files, out_base, class_name, train_ratio=0.7, val_ratio=0.15, seed=42):
    random.seed(seed)
    files = files.copy()
    random.shuffle(files)

    n = len(files)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_files = files[:n_train]
    val_files = files[n_train:n_train+n_val]
    test_files = files[n_train+n_val:]

    split_map = {
        "train": train_files,
        "val": val_files,
        "test": test_files
    }

    for split, split_files in split_map.items():
        for file_path in split_files:
            dst = out_base / split / class_name / file_path.name
            shutil.copy2(file_path, dst)

    return {k: len(v) for k, v in split_map.items()}

clear_split_dir(SPLIT_DIR)
prepare_split_dirs(SPLIT_DIR)

stats_compliant = split_and_copy(compliant_images, SPLIT_DIR, "compliant")
stats_non_compliant = split_and_copy(non_compliant_images, SPLIT_DIR, "not_compliant")

print("Compliant split:", stats_compliant)
print("Not compliant split:", stats_non_compliant)

In [ ]:
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    for cls in ["compliant", "not_compliant"]:
        files = list((SPLIT_DIR / split / cls).glob("*"))
        print(cls, len(files))

## 4. Model trainen

In [ ]:
model = YOLO(MODEL_NAME)

results = model.train(
    data=str(SPLIT_DIR),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project="runs_ibcs",
    name="yolo_cls_ibcs",
    exist_ok=True
)

## 5. Model valideren

In [ ]:
best_model_path = Path("runs_ibcs") / "yolo_cls_ibcs" / "weights" / "best.pt"
model = YOLO(best_model_path)

metrics = model.val(
    data=str(SPLIT_DIR),
    imgsz=IMG_SIZE,
    device=DEVICE
)

print(metrics)

## 6. Voorspellingen maken op testset

In [ ]:
test_images = []
true_labels = []

for cls_name, label_num in [("compliant", 0), ("not_compliant", 1)]:
    files = list((SPLIT_DIR / "test" / cls_name).glob("*"))
    test_images.extend(files)
    true_labels.extend([label_num] * len(files))

print("Aantal testafbeeldingen:", len(test_images))

In [ ]:
pred_labels = []
class_names = ["compliant", "not_compliant"]

for img_path in test_images:
    result = model.predict(source=str(img_path), imgsz=IMG_SIZE, verbose=False)[0]
    pred_idx = int(result.probs.top1)
    pred_labels.append(pred_idx)

print("Voorspellingen klaar.")

## 7. Evaluatie

In [ ]:
print(classification_report(true_labels, pred_labels, target_names=class_names))

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot()
plt.show()

## 8. Een paar losse voorbeelden bekijken

In [ ]:
sample_test = random.sample(test_images, min(6, len(test_images)))

plt.figure(figsize=(14, 8))
for i, img_path in enumerate(sample_test, start=1):
    result = model.predict(source=str(img_path), imgsz=IMG_SIZE, verbose=False)[0]
    pred_idx = int(result.probs.top1)
    pred_name = class_names[pred_idx]
    confidence = float(result.probs.top1conf)

    img = Image.open(img_path)
    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title(f"Pred: {pred_name}\nConf: {confidence:.2f}", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 9. Simpele uitleg genereren

Dit is nog geen echte IBCS-regelverklaring, maar wel een nette basis voor jullie project.

In [ ]:
def explain_prediction(pred_name, confidence):
    if pred_name == "compliant":
        return f"Het model verwacht dat dit dashboard compliant is. Confidence: {confidence:.2f}."
    else:
        return f"Het model verwacht dat dit dashboard not compliant is. Confidence: {confidence:.2f}. Controleer vooral schaalgebruik, consistentie en leesbaarheid."

for img_path in sample_test[:3]:
    result = model.predict(source=str(img_path), imgsz=IMG_SIZE, verbose=False)[0]
    pred_idx = int(result.probs.top1)
    pred_name = class_names[pred_idx]
    confidence = float(result.probs.top1conf)

    print(img_path.name)
    print(explain_prediction(pred_name, confidence))
    print("-" * 50)

## 10. Volgende stap voor jullie project

Jullie huidige repo is nu het best geschikt voor **classification**.

### Als jullie echt YOLO detection willen gebruiken
Dan moeten jullie dashboard-afbeeldingen annoteren met bounding boxes voor bijvoorbeeld:
- chart
- x_axis
- y_axis
- title
- legend
- label
- tick_mark
- scale_text

Pas daarna kun je met YOLO echt onderdelen van een dashboard laten detecteren.

### Slimme aanpak
- **Nu**: classification trainen met jullie bestaande dataset
- **Later**: detection dataset maken voor uitleg per dashboard-element